In [10]:
from IPython.display import clear_output
import numpy as np
import sys, random, time, math, os, copy, pickle
import pygame, argparse
import matplotlib.pyplot as plt
from IPython.display import clear_output
import cv2
np.set_printoptions(suppress=True)

def xy_state(x1, y1, x2, y2):
    d_x = np.sign(x1 - x2)
    d_y = np.sign(y1 - y2)
    n = 3 * d_x + d_y + 4
    if n > 4:
        n -= 1
    return n

def softmax(vect: list):
    vect = np.array(vect, dtype=float)
    vect = np.exp(vect - np.max(vect))
    return vect / np.sum(vect)

dist_names = ['left', 'up', 'right', 'down']
dict_vect = {'left': [-1, 0], 'up': [0, -1], 'right': [1, 0], 'down': [0, 1]}
moves_dict = {'left': 0, 'up': 1, 'right': 2, 'down': 3}

def get_is_wall(allowed_moves, last_move):
    result = np.zeros((5,))
    for current_move in allowed_moves:
        result[moves_dict[current_move]] = 1
    result[4] = int(last_move in allowed_moves)
    result = 1 - result
    return result

def image_to_state(screenshot):
    surface_ = (screenshot[:, :, 0] == 255) & (screenshot[:, :, 1] == 201) & (screenshot[:, :, 2] == 0)
    surface_ = surface_.astype(np.uint8) * 255
    surface_[580:, :] = 0
    contours, _ = cv2.findContours(surface_, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) > 0:
        largest_contour = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest_contour)
        pacman_x = x + w // 2
        pacman_y = y + h // 2

    surface_ = (screenshot[:, :, 0] == 255) & (screenshot[:, :, 1] == 255) & (screenshot[:, :, 2] == 255)
    surface_ = surface_.astype(np.uint8) * 255
    surface_[:30, :] = 0
    y, x = np.meshgrid(np.arange(surface_.shape[0]), np.arange(surface_.shape[1]), indexing='ij')
    gaussian = np.exp(-(((x - pacman_x)**2 / 200**2) + ((y - pacman_y)**2 / 200**2)))
    gaussian /= np.max(gaussian)
    surface_ = surface_.astype(np.double) * gaussian
    flattened_index = surface_.argmax()
    pg_y, pg_x = np.unravel_index(flattened_index, surface_.shape)

    surface_ = (screenshot[:, :, 0] == 2) & (screenshot[:, :, 1] == 17) & (screenshot[:, :, 2] == 179)
    surface_ = surface_.astype(np.uint8) * 255
    output_image = cv2.cvtColor(surface_.astype(np.uint8), cv2.COLOR_GRAY2RGB)
    mask_ = np.zeros((surface_.shape[0], surface_.shape[1]), dtype=np.uint8)
    wsz = 9
    mask_[pacman_y-wsz*4:pacman_y-wsz*2, pacman_x-wsz:pacman_x+wsz] = 1
    mask_[pacman_y+wsz*2:pacman_y+wsz*4, pacman_x-wsz:pacman_x+wsz] = 1
    mask_[pacman_y-wsz:pacman_y+wsz, pacman_x-wsz*4:pacman_x-wsz*2] = 1
    mask_[pacman_y-wsz:pacman_y+wsz, pacman_x+wsz*2:pacman_x+wsz*4] = 1
    surface_ *= mask_

    allowed_moves_ = []
    if np.sum(surface_[pacman_y-wsz*4:pacman_y-wsz*2, pacman_x-wsz:pacman_x+wsz]) == 0 and pacman_y-wsz*4 > 0:
        allowed_moves_.append('up')
    if np.sum(surface_[pacman_y+wsz*2:pacman_y+wsz*4, pacman_x-wsz:pacman_x+wsz]) == 0 and pacman_y+wsz*4 < surface_.shape[0]:
        allowed_moves_.append('down')
    if np.sum(surface_[pacman_y-wsz:pacman_y+wsz, pacman_x-wsz*4:pacman_x-wsz*2]) == 0 and pacman_x-wsz*4 > 0:
        allowed_moves_.append('left')
    if np.sum(surface_[pacman_y-wsz:pacman_y+wsz, pacman_x+wsz*2:pacman_x+wsz*4]) == 0 and pacman_x+wsz*4 < surface_.shape[1]:
        allowed_moves_.append('right')
                
    current_state = xy_state(pacman_x, pacman_y, pg_x, pg_y) * 3 + len(allowed_moves_) - 2
    return current_state

LEVELS = {}

LEVELS[1] = {
    'MODES': (
        [ 'chase', 7 ],
        [ 'chase', 20 ],
        [ 'chase', 7 ],
        [ 'chase', 20 ],
        [ 'chase', 7 ],
        [ 'chase', 20 ],
        [ 'chase', 5 ],
        [ 'chase', 9999999 ]
    ),
    'red_dots_remaining': 20,
    'bonus': 'cherry'
}

FRUITS = {  "cherry": {  'id': 7, "score": 100 },
            "strawberry": {  'id': 8, "score": 300 },
            "orange": {  'id': 9, "score": 500 },
            "apple": {  'id': 10, "score": 700 },
            "melon": {  'id': 11, "score": 1000 },
            "galboss": {  'id': 12, "score": 2000 },
            "bell": {  'id': 13, "score": 3000 },
            "key": {  'id': 14, "score": 5000 }
}

SCATTER = { "red": (26,1) ,
            "blue": (26,29),
            "yellow": (1,29),
            "pink": (1,1)
}

JAIL = { "red": (12,14) ,
         "blue": (13,14),
         "yellow": (15,14),
         "pink": (14,14)
}


PACMAN_TIMERS = {
    "normal": 999999999,
    "chase": 6
}

PACMAN_POS = (13, 23)

FORBIDDEN_UP = [
    (12, 10),
    (15, 10),
    (12, 22),
    (15, 22)
]

# 28 (0-27) x 31 (0-30)
MAP_global = [
        [52, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 53, 52, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 53],
        [50,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 33, 33,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 51],
        [50,  1, 34, 32, 32, 35,  1, 34, 32, 32, 32, 35,  1, 33, 33,  1, 34, 32, 32, 32, 35,  1, 34, 32, 32, 35,  1, 51],
        [50,  1, 33,  0,  0, 33,  1, 33,  0,  0,  0, 33,  1, 33, 33,  1, 33,  0,  0,  0, 33,  1, 33,  0,  0, 33,  1, 51],
        [50,  1, 36, 32, 32, 37,  1, 36, 32, 32, 32, 37,  1, 36, 37,  1, 36, 32, 32, 32, 37,  1, 36, 32, 32, 37,  1, 51],
        [50,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 51],
        [50,  1, 34, 32, 32, 35,  1, 34, 35,  1, 34, 32, 32, 32, 32, 32, 32, 35,  1, 34, 35,  1, 34, 32, 32, 35,  1, 51],
        [50,  1, 36, 32, 32, 37,  1, 33, 33,  1, 36, 32, 32, 35, 34, 32, 32, 37,  1, 33, 33,  1, 36, 32, 32, 37,  1, 51],
        [50,  1,  1,  1,  1,  1,  1, 33, 33,  1,  1,  1,  1, 33, 33,  1,  1,  1,  1, 33, 33,  1,  1,  1,  1,  1,  1, 51],
        [54, 49, 49, 49, 49, 57,  1, 33, 36, 32, 32, 35,  0, 33, 33,  0, 34, 32, 32, 37, 33,  1, 56, 49, 49, 49, 49, 55],
        [16, 16, 16, 16, 16, 50,  1, 33, 34, 32, 32, 37,  0, 36, 37,  0, 36, 32, 32, 35, 33,  1, 51, 16, 16, 16, 16, 16],
        [16, 16, 16, 16, 16, 50,  1, 33, 33,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0, 33, 33,  1, 51, 16, 16, 16, 16, 16],
        [16, 16, 16, 16, 16, 50,  1, 33, 33,  0, 56, 49, 49, 0, 0, 49, 49, 57,  0, 33, 33,  1, 51, 16, 16, 16, 16, 16],
        [48, 48, 48, 48, 48, 58,  1, 36, 37,  0, 51, 64, 64,  0,  0, 64, 64, 50,  0, 36, 37,  1, 59, 48, 48, 48, 48, 48],
        [15, 15, 15, 15, 15, 15,  1,  0,  0,  0, 51, 64,  0,  0,  0,  0, 64, 50,  0,  0,  0,  1, 15, 15, 15, 15, 15, 15],
        [49, 49, 49, 49, 49, 57,  1, 34, 35,  0, 51, 64, 64, 64, 64, 64, 64, 50,  0, 34, 35,  1, 56, 49, 49, 49, 49, 49],
        [16, 16, 16, 16, 16, 50,  1, 33, 33,  0, 59, 48, 48, 48, 48, 48, 48, 58,  0, 33, 33,  1, 51, 16, 16, 16, 16, 16],
        [16, 16, 16, 16, 16, 50,  1, 33, 33,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0, 33, 33,  1, 51, 16, 16, 16, 16, 16],
        [16, 16, 16, 16, 16, 50,  1, 33, 33,  0, 34, 32, 32, 32, 32, 32, 32, 35,  0, 33, 33,  1, 51, 16, 16, 16, 16, 16],
        [52, 48, 48, 48, 48, 58,  1, 36, 37,  0, 36, 32, 32, 35, 34, 32, 32, 37,  0, 36, 37,  1, 59, 48, 48, 48, 48, 53],
        [50,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 33, 33,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 51],
        [50,  1, 34, 32, 32, 35,  1, 34, 32, 32, 32, 35,  1, 33, 33,  1, 34, 32, 32, 32, 35,  1, 34, 32, 32, 35,  1, 51],
        [50,  1, 36, 32, 35, 33,  1, 36, 32, 32, 32, 37,  1, 36, 37,  1, 36, 32, 32, 32, 37,  1, 33, 34, 32, 37,  1, 51],
        [50,  1,  1,  1, 33, 33,  1,  1,  1,  1,  1,  1,  1,  0,  0,  1,  1,  1,  1,  1,  1,  1, 33, 33,  1,  1,  1, 51],
        [54, 32, 35,  1, 33, 33,  1, 34, 35,  1, 34, 32, 32, 32, 32, 32, 32, 35,  1, 34, 35,  1, 33, 33,  1, 34, 32, 55],
        [52, 32, 37,  1, 36, 37,  1, 33, 33,  1, 36, 32, 32, 35, 34, 32, 32, 37,  1, 33, 33,  1, 36, 37,  1, 36, 32, 53],
        [50,  1,  1,  1,  1,  1,  1, 33, 33,  1,  1,  1,  1, 33, 33,  1,  1,  1,  1, 33, 33,  1,  1,  1,  1,  1,  1, 51],
        [50,  1, 34, 32, 32, 32, 32, 37, 36, 32, 32, 35,  1, 33, 33,  1, 34, 32, 32, 37, 36, 32, 32, 32, 32, 35,  1, 51],
        [50,  1, 36, 32, 32, 32, 32, 32, 32, 32, 32, 37,  1, 36, 37,  1, 36, 32, 32, 32, 32, 32, 32, 32, 32, 37,  1, 51],
        [50,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1 , 1 ,51],
        [54, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 55]
]

BLACK = (0, 0, 0)
WHITE = (255, 255, 255)

class Pacman(pygame.sprite.Sprite):
    def __init__(self, my_game, x, y, input_Q):
        pygame.sprite.Sprite.__init__(self)
        self.game = my_game
        self.x = None
        self.y = None
        self.real_x = None
        self.real_y = None
        self.speed = None
        self.mode = None
        self.mode_changed = None
        self.direction = None
        self.image = None
        self.allowed_moves = None
        self.count_moves = None
        self.start_time = None
        self.miss_loops = None
        self.set_bonus = None
        self.reinit(x, y)
        self.last_move = 'left'
        self.Q = input_Q
        self.current_state = 0
        self.action = 0
        self.old_dist = -1
        self.old_dist_ghost = -1
        self.last_moved = False
        self.reward = 0
        self.last_coordinates = None
        self.pacman_time = 0
        self.last_ate_time = 0
        self.last_nearest_pg = None
        self.escape_mode = False  # Режим побега от призраков
        self.escape_counter = 0
        self.radius = 10  # УВЕЛИЧИВАЕМ РАДИУС ДЛЯ СТОЛКНОВЕНИЙ
        self.last_direction_change_time = 0
        self.direction_change_cooldown = 0.5  # 0.5 секунды между сменами направления
        
    def reinit(self, x, y):
        self.x = x
        self.y = y
        self.image = self.game.Pacman_pics['left'][1]
        self.rect = self.image.get_rect()
        self.rect.center = (self.x * 24 + 12, self.y * 24 + 12)
        self.real_x = self.x * 24 * 10
        self.real_y = self.y * 24 * 10
        self.speed = 55
        self.direction = "left"
        self.mode = "normal"
        self.allowed_moves = []
        self.count_moves = 0
        self.miss_loops = 0
        self.start_time = time.time()
        self.mode_changed = False
        self.radius = 3
        self.escape_mode = False
        self.escape_counter = 0
        self.old_dist = -1
        self.old_dist_ghost = -1

    def get_q_table(self):
        return self.Q

    def get_allowed_moves(self):
        """
        Check if pacman is allowed to move here
        """
        global MAP
        self.allowed_moves = []

        # check walls
        if MAP[self.y][self.x-1] < 16:
            self.allowed_moves.append("left")
        if (self.x+1 < 28 and MAP[self.y][self.x+1] < 16) or (self.x == 27 and self.direction == "right"):
            self.allowed_moves.append("right")
        if MAP[self.y - 1][self.x] < 16:
            self.allowed_moves.append("up")
        if MAP[self.y + 1][self.x] < 16:
            self.allowed_moves.append("down")

    def check_pacgums(self):
        global MAP
        chase = False
        if MAP[self.y][self.x] == 1:
            self.game.score += 10
            MAP[self.y][self.x] = 0
            self.game.pacgums -= 1
            self.miss_loops = 5
            self.last_ate_time = self.pacman_time
            return True  # Съели точку
        return chase

    def change_mode(self):
        """
        Two modes for pacman: chase or normal
        """
        self.mode_changed = False
        current_time = time.time()    
        # Enter chase mode
        if self.check_pacgums():
            self.mode = "chase"
            self.start_time = current_time
            self.mode_changed = True
            # УБРАЛИ - призраки не должны убегать
            # for ghost in self.game.Ghosts:
            #     if ghost.mode != "eaten":
            #         ghost.change_mode("runaway")
    
        else:
            mode_time = PACMAN_TIMERS[self.mode]
            if current_time - self.start_time > mode_time:
                if self.mode == "chase":
                    # Reinit ghosts in a row counter
                    self.game.ghosts_in_a_row = 0
                    self.mode = "normal"
                self.start_time = current_time
                self.mode_changed = True

    def find_nearest_pacgum(self):
        """
        Находит ближайшую точку для сбора
        Возвращает координаты и расстояние
        """
        global MAP
        nearest_dist = float('inf')
        nearest_point = None
        
        for y in range(len(MAP)):
            for x in range(len(MAP[0])):
                if MAP[y][x] == 1:
                    dist = abs(x - self.x) + abs(y - self.y)  # Манхэттенское расстояние
                    if dist < nearest_dist:
                        nearest_dist = dist
                        nearest_point = (x, y)
        
        return nearest_point, nearest_dist

    def update(self):
        global MAP

        current_time = time.time()
        if current_time - self.last_direction_change_time < self.direction_change_cooldown:
            # Не меняем направление, слишком рано
            if self.direction not in self.allowed_moves:
                # Направление недоступно, останавливаемся
                self.direction = ''
        else:
            # Можно менять направление
            self.last_direction_change_time = current_time
        current_speed = self.speed
        if self.miss_loops:
            current_speed -= self.miss_loops
            self.miss_loops = 0
    
        # if the next move exceeds the next case
        if self.direction == 'left' and self.real_x-current_speed < (self.x-1)*24*10:
            next_speed = self.real_x - (self.x-1)*240
        elif self.direction == 'right' and self.real_x+current_speed > (self.x+1)*24*10:
            next_speed = (self.x+1)*240  - self.real_x
        elif self.direction == 'up' and self.real_y-current_speed < (self.y-1)*24*10:
            next_speed = self.real_y - (self.y-1)*240
        elif self.direction == 'down' and self.real_y+current_speed > (self.y+1)*24*10:
            next_speed = (self.y+1)*240 - self.real_y
        else:
            next_speed = current_speed
    
        next_loops = (next_speed, current_speed-next_speed)
        for speed in next_loops:
            if speed == 0:
                continue
                
            if (self.real_x / 10) % 24 == 0 and (self.real_y / 10) % 24 == 0:
                self.pacman_time += 1
                self.reward = 0
                self.x = int(self.real_x / 10 / 24)
                self.y = int(self.real_y / 10 / 24)
                
                self.change_mode()
                self.get_allowed_moves()
                
                # Проверяем съели ли точку
                if self.game.score > self.game.old_score:
                    self.last_ate_time = self.pacman_time
                    self.reward += self.game.score - self.game.old_score
                    self.game.old_score = self.game.score
    
                # Поиск ближайшей точки
                nearest_point, nearest_dist = self.find_nearest_pacgum()
                
                if nearest_point:
                    pg_x, pg_y = nearest_point
                    current_dist = nearest_dist
                    
                    # Награда за приближение к точке
                    if self.old_dist != -1 and current_dist < self.old_dist:
                        self.reward += (self.old_dist - current_dist) * 2.0
                    
                    self.old_dist = current_dist
                    
                    # Ищем только активных призраков (вышедших из тюрьмы)
                    ghost_list = []
                    for ghost in self.game.Ghosts:
                        # ПРИЗРАК СЧИТАЕТСЯ АКТИВНЫМ, ЕСЛИ ОН ВЫШЕЛ ИЗ ТЮРЬМЫ
                        # Убрали проверку на режим "eaten" - просто смотрим вышел ли он
                        if ghost.has_exited_jail:
                            dist = abs(ghost.x - self.x) + abs(ghost.y - self.y)
                            ghost_list.append({'x': ghost.x, 'y': ghost.y, 'd': dist})
                    
                    if ghost_list:
                        ind = np.argmin([ghost_list[i]['d'] for i in range(len(ghost_list))])
                        ghost_x = ghost_list[ind]['x']
                        ghost_y = ghost_list[ind]['y']
                        current_dist_ghost = ghost_list[ind]['d']
                        
                        # Штраф за приближение к призраку (только для активных призраков)
                        if self.old_dist_ghost != -1 and current_dist_ghost < self.old_dist_ghost:
                            self.reward -= 3 * (self.old_dist_ghost - current_dist_ghost)
                        
                        self.old_dist_ghost = current_dist_ghost
                    else:
                        # Если нет активных призраков, устанавливаем большое расстояние
                        self.old_dist_ghost = float('inf')
                        ghost_x, ghost_y = 0, 0  # Устанавливаем координаты по умолчанию
                    
                    # Обновляем Q-таблицу
                    self.Q[self.current_state, self.action] = 0.99 * self.Q[self.current_state, self.action] + 0.01 * self.reward
                    
                    # Вычисляем состояние
                    # Если нет активных призраков, используем нейтральное состояние
                    if not ghost_list:
                        ghost_state = 4  # Нейтральное состояние (призрак далеко)
                    else:
                        ghost_state = xy_state(self.x, self.y, ghost_x, ghost_y)
                    
                    pg_state = xy_state(self.x, self.y, pg_x, pg_y)
                    current_state = (ghost_state * 8 + pg_state) * 3 + len(self.allowed_moves) - 2
                    
                    # Проверяем границы состояния
                    if current_state >= self.Q.shape[0]:
                        current_state = self.Q.shape[0] - 1
                    elif current_state < 0:
                        current_state = 0
                        
                    self.current_state = current_state
                    self.is_wall = get_is_wall(self.allowed_moves, self.last_move)
                    
                    # Выбираем действие с помощью softmax, но только если есть доступные ходы
                    if len(self.allowed_moves) > 0:
                        q_prob = softmax(self.Q[self.current_state, :] - 1e6 * self.is_wall)
                        self.action = np.random.choice(len(q_prob), p=q_prob)
                        
                        # Проверяем, что выбранное действие допустимо
                        if self.action < 4:
                            chosen_direction = dist_names[self.action]
                            if chosen_direction in self.allowed_moves:
                                self.direction = chosen_direction
                            else:
                                # Если выбранное направление недоступно, выбираем случайное доступное
                                if self.allowed_moves:
                                    self.direction = random.choice(self.allowed_moves)
                        else:
                            # Остаться на месте
                            # self.last_moved = False
                            pass
                    else:
                        # Нет доступных ходов
                        self.direction = ''
                        self.last_moved = False
    
            # Ручное управление (для отладки)
            keys = pygame.key.get_pressed()
            if keys[pygame.K_LEFT]:
                if MAP[self.y][self.x-1] < 16:
                    self.direction = "left"
            elif keys[pygame.K_RIGHT]:
                if self.x+1 < 28 and MAP[self.y][self.x+1] < 16:
                    self.direction = "right"
            elif keys[pygame.K_UP]:
                if MAP[self.y - 1][self.x] < 16:
                    self.direction = "up"
            elif keys[pygame.K_DOWN]:
                if self.y + 1 < 30 and MAP[self.y + 1][self.x] < 16:
                    self.direction = "down"
    
            # Direction is set : move the ghost
            self.last_moved = True
            if self.direction == "left" and "left" in self.allowed_moves:
                self.last_move = 'left'
                self.real_x -= speed
                self.rect.x = round(self.real_x/10)
                # go to right tunnel 
                if self.rect.x <= -24 :
                    self.rect.x = self.game.WIDTH-24
                    self.real_x = self.rect.x*10
            elif self.direction == "right" and "right" in self.allowed_moves:
                self.last_move = 'right'
                self.real_x += speed
                self.rect.x = round(self.real_x/10)
                # go to left tunnel
                if self.rect.x >= self.game.WIDTH:
                    self.rect.x = -24
                    self.real_x = -240
            elif self.direction == "up" and "up" in self.allowed_moves:
                self.last_move = 'up'
                self.real_y -= speed
                self.rect.y = round(self.real_y/10)
            elif self.direction == "down" and "down" in self.allowed_moves:
                self.last_move = 'down'
                self.real_y += speed
                self.rect.y = round(self.real_y/10)
            else:
                self.last_moved = False
                self.last_move = ''
        if self.last_moved:
            self.count_moves += 1
        self.rect.x = round(self.real_x/10) - 4
        self.rect.y = round(self.real_y/10) - 4
        if self.direction:
            if self.last_moved:
                self.image = self.game.Pacman_pics[self.direction][self.count_moves % 3 + 1]
            else:
                self.image = self.game.Pacman_pics[self.direction][2]

class Ghost(pygame.sprite.Sprite):
    """
    Defines a ghost
    """
    moves = ["left", "right", "up", "down"]
    opposite = ["right", "left", "down", "up"]

    def __init__(self, my_game, x, y, color, mode):
        pygame.sprite.Sprite.__init__(self)
        self.game = my_game
        self.x = None
        self.y = None
        self.real_x = None
        self.real_y = None
        self.color = color
        self.mode = None
        self.old_mode = None
        self.distances = None
        self.allowed_moves = None
        self.forbid_turnback = None
        self.in_tunnel = None
        self.direction = None
        self.speed = None
        self.count_moves = None
        self.blinking_tempo = None
        self.mode_changed = None
        self.start_time = None
        self.image = None
        self.target = None
        self.jail_timer = 0
        self.jail_exit_point = (14, 11)  # Точка выхода из тюрьмы
        self.has_exited_jail = False
        
        # Новые параметры для цикличной смены режимов
        self.mode_cycle_start = time.time()
        self.current_mode_duration = 0
        self.is_in_scatter = False
        self.scatter_target = SCATTER[color]  # Угол для scatter режима
        
        # Время для разных режимов (в секундах) - сделаем более случайными
        self.scatter_durations = [random.uniform(5, 8), random.uniform(5, 8), 
                                  random.uniform(4, 6), random.uniform(4, 6)]
        self.chase_durations = [random.uniform(18, 25), random.uniform(18, 25), 
                                random.uniform(15, 22), random.uniform(100, 200)]  # Последний долгий

# Новые параметры для случайного поведения
        self.random_walk_counter = 0
        self.random_walk_duration = random.randint(20, 60)  # Количество шагов для случайного блуждания
        self.scatter_targets = {
            "red": [(26, 1), (26, 5), (20, 5), (20, 1)],  # Несколько точек в углу
            "pink": [(1, 1), (5, 1), (5, 5), (1, 5)],
            "blue": [(26, 29), (26, 25), (20, 25), (20, 29)],
            "yellow": [(1, 29), (5, 29), (5, 25), (1, 25)]
        }
        self.current_scatter_target_idx = 0
        self.target_change_counter = 0
        self.target_change_frequency = random.randint(40, 80)
        
        self.reinit(x, y, mode)
        
        # self.reinit(x, y, mode)

    def reinit(self, x, y, mode):
        self.x = x
        self.y = y
        self.mode = mode
        self.old_mode = ""
        self.distances = dict()
        self.allowed_moves = []
        self.forbid_turnback = True
        self.direction = ''
        self.speed = 55
        self.blinking_tempo = 0
        self.in_tunnel = False
        self.radius = 10  # УВЕЛИЧИВАЕМ РАДИУС ДЛЯ СТОЛКНОВЕНИЙ
        self.start_time = time.time()
        self.mode_changed = False
        self.jail_timer = 0

        # Инициализация случайных параметров
        self.random_walk_counter = 0
        self.random_walk_duration = random.randint(20, 60)
        self.current_scatter_target_idx = 0
        self.target_change_counter = 0
        self.target_change_frequency = random.randint(40, 80)
        
        # ВАЖНО: сбрасываем флаг выхода из тюрьмы, если режим jail
        if mode == "jail":
            self.has_exited_jail = False
        else:
            self.has_exited_jail = True  # Если не в тюрьме, считаем что вышел
        
        # Инициализация циклических режимов (только для вышедших из тюрьмы)
        if mode != "jail":
            self.mode_cycle_start = time.time()
            self.is_in_scatter = True  # Начинаем с scatter режима
            self.current_mode_duration = self.scatter_durations[0]
        
        if self.color in ('red', 'yellow', 'blue', 'pink'):
            self.image = self.game.Ghost_pics[self.color]['left'][1]
        else:
            self.image = pygame.Surface((32, 32))
            self.image.fill(WHITE)
        
        self.rect = self.image.get_rect()
        self.rect.center = (self.x * 24 + 12, self.y * 24 + 12)
        self.real_x = self.x * 24 * 10
        self.real_y = self.y * 24 * 10

    def update_mode_cycle(self):
        """Обновляет циклическую смену режимов scatter/chase"""
        """Обновляет циклическую смену режимов scatter/chase"""
        if self.mode in ["jail", "eaten"] or not self.has_exited_jail:
            return  # В этих режимах своя логика
            
        current_time = time.time()
        elapsed = current_time - self.mode_cycle_start
        
        # Проверяем, не пора ли сменить режим
        if elapsed > self.current_mode_duration:
            if self.is_in_scatter:
                # Заканчивается scatter, начинается chase
                self.is_in_scatter = False
                self.mode = "chase"
                # Случайная длительность chase режима
                level_idx = min(self.game.level - 1, 3)
                base_duration = self.chase_durations[level_idx]
                self.current_mode_duration = random.uniform(base_duration * 0.7, base_duration * 1.3)
            else:
                # Заканчивается chase, начинается scatter
                self.is_in_scatter = True
                self.mode = "scatter"
                # Случайная длительность scatter режима
                level_idx = min(self.game.level - 1, 3)
                base_duration = self.scatter_durations[level_idx]
                self.current_mode_duration = random.uniform(base_duration * 0.7, base_duration * 1.3)
            
            self.mode_cycle_start = current_time
            # Разрешаем разворот при смене режима
            self.forbid_turnback = False
            self.mode_changed = True
            
            # При смене режима обновляем параметры случайного блуждания
            self.random_walk_counter = 0
            self.random_walk_duration = random.randint(20, 60)

    def choose_direction(self):
        self.distances = dict()
        
        if self.forbid_turnback == False:
            self.forbid_turnback = True
        
        # Если призрак в тюрьме и должен выйти
        if self.mode == "jail" and self.has_exited_jail:
            # Определяем точку выхода из тюрьмы
            jail_exit = (14, 11)  # Координаты выхода
            
            # Если мы уже на выходе, выходим
            if (self.x, self.y) == (14, 14):
                # Мы в центре тюрьмы, идем вверх к выходу
                self.target = (14, 11)
            elif self.y > 14:
                # Мы внизу тюрьмы, идем в центр
                self.target = (14, 14)
            elif self.x != 14:
                # Мы не по центру, идем к центру
                self.target = (14, self.y)
            else:
                # Идем вверх к выходу
                self.target = (14, 11)
        elif self.mode == "scatter":
            # В scatter режиме идем в свой угол
            self.target = self.scatter_target
        elif self.mode == "chase":
            # В chase режиме преследуем пакмана
            self.target = (self.game.pacman.x, self.game.pacman.y)
        else:
            # По умолчанию
            self.target = (self.game.pacman.x, self.game.pacman.y)
        
        # Рассчитываем расстояние для каждого возможного направления
        for direction in self.allowed_moves:
            x = self.x
            y = self.y
            if direction == "up":
                y = self.y - 1
            elif direction == "down":
                y = self.y + 1
            elif direction == "left":
                x = self.x - 1
            else:
                x = self.x + 1
                
            dist_x = abs(x - self.target[0])
            dist_y = abs(y - self.target[1])
            distance = math.sqrt(dist_x**2 + dist_y**2)
            self.distances[direction] = distance
                
        # Выбираем направление с минимальным расстоянием
        min_dist = float('inf')
        best_dir = self.direction
        
        for direction, distance in self.distances.items():
            if distance < min_dist:
                min_dist = distance
                best_dir = direction
                    
        self.direction = best_dir

    def get_random_target(self):
        """Возвращает случайную точку на карте"""
        # Избегаем тюрьмы и туннелей
        possible_targets = []
        for y in range(1, 29):  # Избегаем краев
            for x in range(1, 27):  # Избегаем краев
                if MAP[y][x] < 16 and (x, y) not in [(13, 14), (14, 14), (12, 14), (15, 14)]:
                    possible_targets.append((x, y))
        
        if possible_targets:
            return random.choice(possible_targets)
        return (self.x, self.y)  # На всякий случай

    def get_random_patrol_target(self):
        """Возвращает точку для патрулирования"""
        # Патрулируем центральную часть карты
        patrol_points = [
            (13, 11), (14, 11),  # Выход из тюрьмы
            (13, 17), (14, 17),  # Центр
            (10, 14), (17, 14),  # По бокам
            (13, 20), (14, 20)   # Низ
        ]
        return random.choice(patrol_points)
    
    # Checks the free positions around the ghost
    def get_allowed_moves(self, x=None, y=None):
        global MAP
        allowed_moves = []
    
        if not x:
            x = self.x
            y = self.y
    
        # Призраки могут ходить через ворота тюрьмы (значение 0)
        if MAP[y][x-1] < 16 or MAP[y][x-1] == 0:
            allowed_moves.append("left")
        if (x+1 < 28 and (MAP[y][x+1] < 16 or MAP[y][x+1] == 0)) or x+1 == 28:
            allowed_moves.append("right")
        
        # Движение вверх/вниз
        if y > 0 and (MAP[y - 1][x] < 16 or MAP[y - 1][x] == 0):
            allowed_moves.append("up")
        if y < 30 and (MAP[y + 1][x] < 16 or MAP[y + 1][x] == 0):
            allowed_moves.append("down")
    
        if self.direction != '':
            reverse = self.opposite[self.moves.index(self.direction)]
    
            if self.forbid_turnback and reverse in allowed_moves:
                allowed_moves.remove(reverse)
    
            # It should happen only if a row line
            if len(allowed_moves) == 0:
                allowed_moves.append(reverse)
    
        return allowed_moves

    # main function
    def update(self):
        global MAP
            # Обновляем циклическую смену режимов
        self.update_mode_cycle()
        

        if MAP[self.y][self.x] == 15:
            self.in_tunnel = True
        else:
            self.in_tunnel = False
    
        # Проверяем, если призрак в тюрьме но должен выйти
        if self.mode == "jail" and self.has_exited_jail:
            # ПРОСТАЯ ЛОГИКА: если призрак должен выйти, меняем режим
            if (self.x, self.y) == (14, 11):
                # Вышли из тюрьмы - меняем режим
                self.mode = "scatter"
                self.mode_cycle_start = time.time()
                self.is_in_scatter = True
                self.current_mode_duration = self.scatter_durations[0]
                self.forbid_turnback = False
    
        # Отладочный вывод
        # if self.color == 'red' and self.mode == "jail":
        #     print(f"Red ghost: mode={self.mode}, exited={self.has_exited_jail}, pos=({self.x},{self.y})")
        
        current_speed = self.speed
        if self.in_tunnel:
            current_speed = int(self.speed/2)

        # if the next move exceeds the next case
        if self.direction == 'left' and self.real_x-current_speed < (self.x-1)*24*10:
            next_speed = self.real_x - (self.x-1)*240
        elif self.direction == 'right' and self.real_x+current_speed > (self.x+1)*24*10:
            next_speed = (self.x+1)*240  - self.real_x
        elif self.direction == 'up' and self.real_y-current_speed < (self.y-1)*24*10:
            next_speed = self.real_y - (self.y-1)*240
        elif self.direction == 'down' and self.real_y+current_speed > (self.y+1)*24*10:
            next_speed = (self.y+1)*240 - self.real_y
        else:
            next_speed = current_speed
            
        next_loops = (next_speed, current_speed-next_speed)

        for speed in next_loops:
            if speed == 0:
                continue
                
            if (self.real_x /10) % 24 == 0 and (self.real_y/10) % 24 == 0:
                self.x = int(self.real_x / 10 / 24)
                self.y = int(self.real_y / 10 / 24)

                # Проверяем, вышли ли мы из тюрьмы
                if self.mode == "jail" and not self.has_exited_jail:
                    if (self.x, self.y) == (14, 11):
                        self.has_exited_jail = True
                        self.mode = "scatter"
                        self.mode_cycle_start = time.time()
                        self.is_in_scatter = True
                        self.current_mode_duration = self.scatter_durations[0]

                # What are the allowed moves ?
                self.allowed_moves = self.get_allowed_moves()
                self.choose_direction()

                # change mode alreay done, directino alreay set, we can forbid
                self.forbid_turnback = True

            # Direction is set : move the ghost
            if self.direction == "left":
                self.real_x -= speed
                self.rect.x = round(self.real_x/10)
                # go to right tunnel 
                if self.rect.x <= -24 :
                    self.rect.x = self.game.WIDTH-24
                    self.real_x = self.rect.x*10
            elif self.direction == "right":
                self.real_x += speed
                self.rect.x = round(self.real_x/10)
                # go to left tunnel
                if self.rect.x >= self.game.WIDTH:
                    self.rect.x = -24
                    self.real_x = -240
            elif self.direction == "up":
                self.real_y -= speed
                self.rect.y = round(self.real_y/10)
            elif self.direction == "down":
                self.real_y += speed
                self.rect.y = round(self.real_y/10)

        self.rect.x = round(self.real_x/10) - 4
        self.rect.y = round(self.real_y/10) - 4

        # Всегда показываем цветного призрака
        self.image = self.game.Ghost_pics[self.color][self.direction][int(self.blinking_tempo) % 2 + 1]

class Game:
    """
    Main class that manages the full game
    """
    def __init__(self, input_Q, max_loops):
        global MAP
        self.theme = "default"
        self.dymmy = None
        self.Pacman_pics = None
        self.Dead_pacman = None
        self.Ghost_pics = None
        self.Ghost_eyes = None
        self.Scores = None
        self.Frightened_ghost = None
        self.Frightened_ghost_blinking = None
        self.Walls = None
        self.Bonuses = None
        self.Ready = None
        self.ghosts_in_a_row = 0
        self.win = False
        self.trunc = False
        self.max_count_loops = max_loops

        # ghosts are set using setattr but pylint is yelling...
        self.pink = None
        self.red = None
        self.blue = None
        self.yellow = None
        self.munch = None
        self.WIDTH = len(MAP[0])*24
        self.HEIGHT = len(MAP)*24
        self.FULL_WIDTH = self.WIDTH
        self.FULL_HEIGHT = self.HEIGHT + 72
        self.lifes = 3
        self.pacgums = 0
        self.old_score = 0
        self.score = 0
        self.scale = 1
        self.FPS = 1000
        self.count_loops = 0
        self.level = 0
        self.start_mode_timer = None
        self.current_mode_idx = 0
        self.current_mode = None
        pygame.init()
        self.parse_args()
        display_infos = pygame.display.Info()
        y_resolution = display_infos.current_h
        if y_resolution < 800:
            self.scale = 0.75
        self.screen = pygame.display.set_mode((int(self.FULL_WIDTH * self.scale), int(self.FULL_HEIGHT * self.scale)))
        self.fake_screen = pygame.Surface((self.FULL_WIDTH, self.FULL_HEIGHT))
        self.surface = pygame.Surface((self.WIDTH, self.HEIGHT))
        self.top = pygame.Surface((self.WIDTH, 32))
        self.bottom = pygame.Surface((self.WIDTH, 40))
        self.load_bitmaps()
        self.all_sprites = pygame.sprite.Group()
        self.all_ghosts = pygame.sprite.Group()
        self.bonus = None
        
        # Инициализируем pacgums после загрузки MAP
        # Инициализируем pacgums после загрузки MAP
        # Инициализируем pacgums после загрузки MAP
        self.count_pacgums()
        

        # Таймеры для выхода призраков из тюрьмы (в секундах)
        self.ghost_release_timers = {
            "red": 0,      # Красный сразу
            "pink": 5,     # Розовый через 5 секунд
            # "blue": 15,    # Синий через 15 секунд  
            # "yellow": 25   # Желтый через 25 секунд
        }
        
        self.ghost_release_counters = {
            "red": 0,
            "pink": 0,
            # "blue": 0,
            # "yellow": 0
        }
        
        self.ghosts_released = {
            "red": False,
            "pink": False,
            # "blue": False,
            # "yellow": False
        }
        
        # Создаем только ДВА призрака, все начинают в тюрьме
        self.red = Ghost(self, JAIL["red"][0], JAIL["red"][1], 'red', "jail")
        self.pink = Ghost(self, JAIL["pink"][0], JAIL["pink"][1], 'pink', "jail")
        # self.blue = Ghost(self, JAIL["blue"][0], JAIL["blue"][1], 'blue', "jail")
        # self.yellow = Ghost(self, JAIL["yellow"][0], JAIL["yellow"][1], 'yellow', "jail")
        
        # Создаем список только ДВУХ призраков
        self.Ghosts = [self.red, self.pink]
        # self.Ghosts = [self.red, self.pink, self.blue, self.yellow]
        
        # Красный может выйти сразу
        self.red.has_exited_jail = True
        self.red.forbid_turnback = False
        
        self.pacman = Pacman(self, PACMAN_POS[0], PACMAN_POS[1], input_Q)
        self.all_sprites.add(self.pacman)
        self.all_sprites.add(self.Ghosts)
        self.all_ghosts.add(self.Ghosts)

    def parse_args(self):
        parser = argparse.ArgumentParser()
        parser.add_argument('-t', '--theme', help='Theme name', default='default')
        args, unknown = parser.parse_known_args()
        if not os.path.isdir(os.path.join(os.path.dirname(os.path.abspath('')), 'themes/'+args.theme)):
            self.theme = 'default'
        else:
            self.theme = args.theme

    def load_bitmaps(self):
        global MAP
        game_folder = os.getcwd()
        img_folder = os.path.join(game_folder, 'themes\\'+self.theme+'\\img')
        self.Frightened_ghost = dict()
        self.Frightened_ghost[1] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, 'frightened_ghost_1.png')).convert(),(32,32))
        self.Frightened_ghost[2] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, 'frightened_ghost_2.png')).convert(),(32,32))
        self.Frightened_ghost[1].set_colorkey(BLACK)
        self.Frightened_ghost[2].set_colorkey(BLACK)
        self.Frightened_ghost_blinking = dict()
        self.Frightened_ghost_blinking[1] = self.Frightened_ghost[1]
        self.Frightened_ghost_blinking[2] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, 'frightened_ghost_4.png')).convert(),(32,32))
        self.Frightened_ghost_blinking[3] = self.Frightened_ghost[2]
        self.Frightened_ghost_blinking[4] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, 'frightened_ghost_3.png')).convert(),(32,32))
        self.Frightened_ghost_blinking[2].set_colorkey(BLACK)
        self.Frightened_ghost_blinking[4].set_colorkey(BLACK)
        self.Ghost_pics = dict()
        # Оставляем только красного и розового
        for color in ('red', 'pink'):  # УБРАЛИ 'blue', 'yellow'
            self.Ghost_pics[color] = dict()
            for direction in ('left', 'right', 'up', 'down'):
                self.Ghost_pics[color][direction] = dict()
                for i in range(1, 3):
                    self.Ghost_pics[color][direction][i] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, color+'_'+direction+'_ghost_'+str(i)+'.png')).convert(),(32,32))
                    self.Ghost_pics[color][direction][i].set_colorkey(BLACK)
        self.Ghost_eyes = dict()
        for direction in ('left', 'right', 'up', 'down'):
            self.Ghost_eyes[direction] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, 'ghost_eyes_'+direction+'.png')).convert(),(32,32))
            self.Ghost_eyes[direction].set_colorkey(BLACK)
        self.Pacman_pics = dict()
        for direction in ('left', 'right', 'up', 'down'):
            self.Pacman_pics[direction] = dict()
            for i in range(1, 4):
                self.Pacman_pics[direction][i] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, 'pacman_'+direction+'_'+str(i)+'.png')).convert(),(32,32))
                self.Pacman_pics[direction][i].set_colorkey(BLACK)
        self.Dead_pacman = dict()
        for i in range(1, 11):
            self.Dead_pacman[i] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, 'pacman_dead_'+str(i)+'.png')).convert(),(32,32))
            self.Dead_pacman[i].set_colorkey(BLACK)
        self.Scores = dict()
        for score in (100, 200, 300, 400, 500, 700, 800, 1000, 1600, 2000, 3000, 5000):
            self.Scores[score] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, str(score)+'.png')).convert(),(32,32))
            self.Scores[score].set_colorkey(BLACK)
        self.Walls = dict()
        for l in MAP:
            for c in l:
                if c not in self.Walls:
                    png = os.path.join(img_folder, str(c) + ".png")
                    if os.path.exists(png):
                        self.Walls[c] = pygame.image.load(png).convert()
        for bonus in range(7,15):
            self.Walls[bonus] = pygame.image.load(os.path.join(img_folder, str(bonus)+'.png')).convert()
            self.Walls[bonus].set_colorkey(BLACK)
        self.Ready = pygame.image.load(os.path.join(img_folder, 'ready.png')).convert()
        self.Ready.set_colorkey(BLACK)

    def display_board_game(self):
        self.clear_all_surfaces()
        self.display_score()
        self.display_map(self.surface)
        self.all_sprites.draw(self.surface)
        self.display_lifes(self.bottom)
        self.blit_all_surfaces()

    def display_text(self, my_surface, my_text, pos_x, pos_y, color=WHITE):
        font = pygame.font.Font('RetroGaming.ttf', 18)
        text = font.render(my_text, True, color)
        my_surface.blit(text, (pos_x, pos_y))

    def display_score(self):
        self.display_text(self.top, "Player 1     Score: {}".format(self.score), 24, 6)

    def display_lifes(self, my_surface):
        for life in range(0, self.lifes - 1):
            my_surface.blit(self.Pacman_pics['right'][2], (life*32+24, 4))

    def start_game(self):
        i = 0
        while i<4:
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    break
            i += 1 
            time.sleep(0.001)

    def loose_life(self):
        global MAP
        self.lifes -= 1
        # remove bonus
        MAP[17][13] = 0
        i = 1
        self.pacman.Q[self.pacman.current_state, self.pacman.action] = 0.99 * self.pacman.Q[self.pacman.current_state, self.pacman.action] - 0.01 * 1000
        while i < 15:
           
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    break
            self.clear_all_surfaces()
            self.display_map(self.surface)
            if i<11:
                self.surface.blit(self.Dead_pacman[i], (self.pacman.rect.x, self.pacman.rect.y))
            else:
                if i%2:
                    self.surface.blit(self.Dead_pacman[10], (self.pacman.rect.x, self.pacman.rect.y))
            self.display_lifes(self.bottom)
            self.blit_all_surfaces()
            pygame.display.flip()
            i += 1
            time.sleep(0.001)
        pygame.display.update()
        
        # Переинициализируем пакмана
        self.pacman.reinit(PACMAN_POS[0], PACMAN_POS[1])
        
        # Сбрасываем таймеры выпуска призраков
        self.reset_ghost_release()
        
        if self.lifes > 0:
            self.start_game()
        self.current_mode_idx = 0
        self.start_mode_timer = time.time()

    def clear_all_surfaces(self):
        self.fake_screen.fill(BLACK)
        self.surface.fill(BLACK)
        self.top.fill(BLACK)
        self.bottom.fill(BLACK)

    def blit_all_surfaces(self):
        self.fake_screen.blit(self.top, (0, 0))
        self.fake_screen.blit(self.surface, (0, 32))
        self.fake_screen.blit(self.bottom, (0, self.HEIGHT+32))
        self.screen.blit(self.scale_output(self.fake_screen, self.scale), (0, 0))

    def scale_output(self, my_surface, my_scale):
        if my_scale != 1:
            frame = pygame.transform.scale(my_surface, (int(self.FULL_WIDTH * my_scale), int(self.FULL_HEIGHT * my_scale)))
        else:
            frame = my_surface
        return frame

    def count_pacgums(self):
        global MAP
        y_length = len(MAP)
        x_length = len(MAP[0])
        self.pacgums = 0
        for y in range(y_length):
            for x in range(x_length):
                if MAP[y][x] in (1, 2):
                    self.pacgums += 1
        return self.pacgums

    def display_map(self, my_surface):
        global MAP
        y_length = len(MAP)
        x_length = len(MAP[0])
        for y in range(y_length):
            for x in range(x_length):
                c = MAP[y][x]
                if c in self.Walls:
                    if c != 2:
                        my_surface.blit(self.Walls[c], (x*24, y*24))
                    else:
                        if self.count_loops%8 in range(0,3):
                            my_surface.blit(self.Walls[c], (x*24, y*24))

    def collided(self):
        hit_list = pygame.sprite.spritecollide(self.pacman, self.all_ghosts, False, pygame.sprite.collide_circle)
        return hit_list

    def update_ghost_release(self, dt):
        """
        Обновляет таймеры выпуска призраков
        """
        # ТОЛЬКО ДЛЯ ДВУХ ПРИЗРАКОВ
        colors = ['red', 'pink']
        
        for i, color in enumerate(colors):
            ghost = self.Ghosts[i]
            
            # Если призрак еще в тюрьме и не выпущен
            if ghost.mode == "jail" and not ghost.has_exited_jail:
                self.ghost_release_counters[color] += dt
                
                # Если достигли времени выпуска
                if self.ghost_release_counters[color] >= self.ghost_release_timers[color]:
                    print(f"Время вышло! Выпускаем {color} призрака! (таймер: {self.ghost_release_counters[color]:.1f}s)")
                    ghost.has_exited_jail = True
                    # Разрешаем разворот чтобы мог выйти
                    ghost.forbid_turnback = False
                    # Устанавливаем цель - выход из тюрьмы
                    ghost.target = (14, 11)
                    
                    # Также обновляем флаг в словаре
                    self.ghosts_released[color] = True
    
    def release_ghost(self, color):
        """
        Выпускает призрака из тюрьмы
        """
        if self.ghosts_released[color]:
            return  # Уже выпущен
        
        # Находим призрака по цвету
        ghost_to_release = None
        for ghost in self.Ghosts:
            if ghost.color == color and ghost.mode == "jail":
                ghost_to_release = ghost
                break
        
        if ghost_to_release:
            print(f"Выпускаем {color} призрака!")
            # Меняем режим призрака на scatter и разрешаем выход
            ghost_to_release.mode = "scatter"
            ghost_to_release.has_exited_jail = True  # Разрешаем выход
            ghost_to_release.mode_cycle_start = time.time()
            ghost_to_release.is_in_scatter = True
            ghost_to_release.current_mode_duration = ghost_to_release.scatter_durations[0]
            
            # Отмечаем как выпущенного
            self.ghosts_released[color] = True

    
    
    def reset_ghost_release(self):
        """
        Сбрасывает таймеры выпуска призраков (при смерти пакмана)
        """
        # Сбрасываем все таймеры (ТОЛЬКО ДЛЯ ДВУХ ПРИЗРАКОВ)
        self.ghost_release_counters = {
            "red": 0,
            "pink": 0
        }
        
        self.ghosts_released = {
            "red": False,
            "pink": False
        }
        
        # Всех призраков возвращаем в тюрьму
        for ghost in self.Ghosts:
            ghost.reinit(JAIL[ghost.color][0], JAIL[ghost.color][1], "jail")
            ghost.has_exited_jail = False
            ghost.forbid_turnback = True
           
    def play(self):
        self.ghosts_in_a_row = 0
        lives_gained = 0
        clock = pygame.time.Clock()
        running = True
        game_start_time = time.time()
        last_time = game_start_time
        
        # Добавим счетчик столкновений (ТОЛЬКО ДЛЯ ДВУХ ПРИЗРАКОВ)
        collision_counter = {'red': 0, 'pink': 0}
        
        while running and self.level <= 1:
            self.count_loops = 0
            self.start_game()
            self.level += 1
            self.current_mode_idx = 0
            self.current_mode = LEVELS[self.level]['MODES'][self.current_mode_idx][0]
            self.start_mode_timer = time.time()
            
            while running:
                clock.tick(self.FPS)
                self.count_loops += 1
                
                # Вычисляем время, прошедшее с последнего кадра
                current_time = time.time()
                dt = current_time - last_time
                last_time = current_time
                
                # Обновляем выпуск призраков
                self.update_ghost_release(dt)
                
                # Периодически выводим статус призраков
                if self.count_loops % 500 == 0:
                    print(f"\n Статус призраков на итерации {self.count_loops} ")
                    for ghost in self.Ghosts:
                        status = "активен" if ghost.has_exited_jail else "в тюрьме"
                        mode = ghost.mode
                        print(f"{ghost.color}: {status}, режим={mode}, позиция=({ghost.x},{ghost.y})")
                    
                
                for event in pygame.event.get():
                    if event.type == pygame.QUIT:
                        running = False
                current_time = time.time()
                if current_time - self.start_mode_timer > LEVELS[self.level]['MODES'][self.current_mode_idx][1]:
                    self.start_mode_timer = current_time
                    self.current_mode_idx += 1
                    self.current_mode = LEVELS[self.level]['MODES'][self.current_mode_idx][0]
                self.pacman.update()
                self.all_ghosts.update()
                
                if self.count_loops % 2 == 0:
                    self.display_board_game()
                pygame.display.flip()
    
                # Collision test - ПРОСТАЯ ПРОВЕРКА
                for ghost in self.Ghosts:
                    # Проверяем, находятся ли пакмен и призрак в одной клетке
                    if (self.pacman.x == ghost.x and self.pacman.y == ghost.y):
                        # ПАКМЕН УМИРАЕТ ОТ ЛЮБОГО ПРИЗРАКА
                        print(f"\n СМЕРТЬ от {ghost.color} призрака!")
                        print(f"   Позиция: ({ghost.x},{ghost.y})")
                        collision_counter[ghost.color] += 1
                        self.loose_life()
                        break
                
                if self.pacgums == 0:
                    running = False
                    self.win = True
                if self.lifes == 0:
                    running = False
                if self.count_loops > self.max_count_loops:
                    running = False
                    self.trunc = True
        
        return (self.pacman.get_q_table(), self.score, self.count_loops, self.win, self.trunc)

# Root code
def main():
    global MAP
    
    try:
        with open("q_table_with_ghost.pkl", "rb") as file:
            input_q_table = pickle.load(file)
            print("Загружена сохраненная Q-таблица")
    except FileNotFoundError:
        print("Создаем новую Q-таблицу")
        input_q_table = np.ones((24 * 8, 5)) * 0.1
        for i in range(input_q_table.shape[0]):
            input_q_table[i] = np.random.uniform(0.05, 0.2, 5)
        
    for i in range(3):
        MAP = copy.deepcopy(MAP_global)
        game = Game(input_q_table, 10000)  # Увеличил до 10000 итераций
        input_q_table, score_, iteration_, win_, trunc_ = game.play()
        print(f"Игра {i+1}: Очки={score_}, Итерации={iteration_}, Победа={win_}, Прервано={trunc_}")
        pygame.quit()
        
        with open("q_table_with_ghost.pkl", "wb") as file:
            pickle.dump(input_q_table, file)
        print("Q-таблица сохранена")
        
if __name__ == "__main__":
    main()

Загружена сохраненная Q-таблица
Игра 1: Очки=540, Итерации=294, Победа=False, Прервано=False
Q-таблица сохранена
Игра 2: Очки=60, Итерации=27, Победа=False, Прервано=False
Q-таблица сохранена
Игра 3: Очки=70, Итерации=33, Победа=False, Прервано=False
Q-таблица сохранена
